<a href="https://colab.research.google.com/github/ankitaguptacs24-ai/AI_LAB/blob/main/1BF24CS057_Week6_FOL_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def is_variable(expr):
    """Checks if an expression is a variable (typically represented as a string starting with a lowercase letter)."""
    return isinstance(expr, str) and expr.isidentifier() and expr[0].islower()

def is_compound(expr):
    """Checks if an expression is a compound term (list or tuple with a functor and arguments)."""
    return isinstance(expr, (list, tuple)) and len(expr) > 0

def get_functor(expr):
    return expr[0]

def get_args(expr):
    return expr[1:]

def occur_check(var, expr, subst):
    """Returns True if the variable 'var' occurs anywhere inside 'expr' under substitution 'subst'."""
    if var == expr:
        return True
    elif is_variable(expr):
        if expr in subst:
            return occur_check(var, subst[expr], subst)
    elif is_compound(expr):
        return any(occur_check(var, arg, subst) for arg in get_args(expr))
    return False

def unify_var(var, val, subst):
    """Unifies a variable with a value, updating the substitution dictionary."""
    if var in subst:
        return unify(subst[var], val, subst)
    elif is_variable(val) and val in subst:
        return unify(var, subst[val], subst)
    elif occur_check(var, val, subst):
        return None  # Fail due to occur check
    else:
        # Update existing substitution mappings with the new variable assignment
        updated_subst = subst.copy()
        updated_subst[var] = val
        return updated_subst

def unify(x, y, subst=None):
    """Main recursive unification function."""
    if subst is None:
        subst = {}

    if subst is None:  # Failure propagation
        return None

    if x == y:
        return subst
    elif is_variable(x):
        return unify_var(x, y, subst)
    elif is_variable(y):
        return unify_var(y, x, subst)
    elif is_compound(x) and is_compound(y):
        if get_functor(x) != get_functor(y) or len(get_args(x)) != len(get_args(y)):
            return None

        # Recursively unify arguments
        temp_subst = subst
        for arg_x, arg_y in zip(get_args(x), get_args(y)):
            temp_subst = unify(arg_x, arg_y, temp_subst)
            if temp_subst is None:
                return None
        return temp_subst
    else:
        return None

In [ ]:
# Test Cases to Verify Unification

# Representation:
# Constants/Functors start with uppercase: 'A', 'B', 'f', 'g'
# Variables start with lowercase: 'x', 'y', 'z'

test_cases = [
    # 1. Simple matching constants
    ('A', 'A', "Matching constants"),
    # 2. Simple mismatch constants
    ('A', 'B', "Mismatch constants"),
    # 3. Unify variable with constant
    ('x', 'A', "Variable with constant"),
    # 4. Unify compound terms: f(x, B) and f(A, y)
    (['f', 'x', 'B'], ['f', 'A', 'y'], "Compound terms f(x, B) and f(A, y)"),
    # 5. Fail occur check: x and f(x)
    ('x', ['f', 'x'], "Occur check failure"),
    # 6. More complex unification: p(x, g(x)) and p(A, y)
    (['p', 'x', ['g', 'x']], ['p', 'A', 'y'], "Complex terms p(x, g(x)) and p(A, y)")
]

for x, y, desc in test_cases:
    result = unify(x, y)
    print(f"Test: {desc}")
    print(f"  Expression 1: {x}")
    print(f"  Expression 2: {y}")
    print(f"  Result Substitution: {result}\n")

Test: Matching constants
  Expression 1: A
  Expression 2: A
  Result Substitution: {}

Test: Mismatch constants
  Expression 1: A
  Expression 2: B
  Result Substitution: None

Test: Variable with constant
  Expression 1: x
  Expression 2: A
  Result Substitution: {'x': 'A'}

Test: Compound terms f(x, B) and f(A, y)
  Expression 1: ['f', 'x', 'B']
  Expression 2: ['f', 'A', 'y']
  Result Substitution: {'x': 'A', 'y': 'B'}

Test: Occur check failure
  Expression 1: x
  Expression 2: ['f', 'x']
  Result Substitution: None

Test: Complex terms p(x, g(x)) and p(A, y)
  Expression 1: ['p', 'x', ['g', 'x']]
  Expression 2: ['p', 'A', 'y']
  Result Substitution: {'x': 'A', 'y': ['g', 'x']}

